# K-Nearest Neighbors Classifier – Solution

**Short name (GitHub):** `KNN_Cancer`  
**Lab source:** Codecademy *K-Nearest Neighbors Classifier* (movies from-scratch) + *Cancer Classifier* (sklearn)  
**Language:** Python (NumPy + pandas + Matplotlib + scikit-learn)

Worked answers. Compare with your `KNN_Cancer_Practice_Skeleton.ipynb` cells.  
Companion files: `KNN_Cancer_Cheatsheet.docx`, `KNN_Cancer_Reusable_Template.ipynb`, `knn_cancer_flowchart.png`, `KNN_Cancer.py`.

### Learning objectives
- Compute Euclidean distance in 2-D and in *n*-D
- Min-max normalize so budget does not drown year / duration
- Find the *k* nearest labeled movies and majority-vote good vs bad
- Load the Wisconsin breast-cancer set, split train/valid, fit `KNeighborsClassifier`
- Sweep *k* and plot validation accuracy (bias–variance)
- Alternate implementations (broadcast NumPy, Manhattan, `NearestNeighbors`)
- Extra practice on a 2-D blob set and a DTI/utilization loan book
- Monte-Carlo: *k*, label noise, sample size, extra noise features
- Rewrite the same result for an analyst, a medical director, a patient, a non-specialist

### Data files
- `data/knn_movies.csv` — 40 films (duration, year, budget, good)
- `data/knn_cancer.csv` — 569 cells × 30 features + target (0 = malignant, 1 = benign)
- `data/knn_2d.csv` — synthetic 2-feature set for boundaries
- `data/knn_loans.csv` — DTI + utilization → default

### Flowchart
Open `knn_cancer_flowchart.png` while you work.


## Inline cheat-sheet (keep this cell visible)

See also **`KNN_Cancer_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Euclidean | $d(a,b)=\sqrt{\sum_j (a_j-b_j)^2}$ |
| Manhattan | $d_1(a,b)=\sum_j \|a_j-b_j\|$ |
| Min-max | $x'=(x-x_{\min})/(x_{\max}-x_{\min})$ |
| Vote | predict the majority class among the $k$ nearest |
| Ties | prefer odd $k$, else use the closest neighbor |
| Overfit | $k$ too small → outliers dominate |
| Underfit | $k$ too large → vote ≈ global majority |
| sklearn | `KNeighborsClassifier(n_neighbors=k).fit(X,y).score(Xv,yv)` |
| Split | `train_test_split(..., test_size=0.2, random_state=100, stratify=y)` |
| Cancer labels | `target` 0 = malignant, 1 = benign |

**Flow:** features → scale → split → distance → $k$ neighbors → vote → sweep $k$ → simulate.


## 0. Packages


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix

%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("Libraries loaded")


## 1. Distance between points (2-D)


In [ ]:
star_wars = [125, 1977]
raiders = [115, 1981]
mean_girls = [97, 2004]

def distance_2d(movie1, movie2):
    length_difference = (movie1[0] - movie2[0]) ** 2
    year_difference = (movie1[1] - movie2[1]) ** 2
    return (length_difference + year_difference) ** 0.5

print(distance_2d(star_wars, raiders))
print(distance_2d(star_wars, mean_girls))
print("closer to Star Wars:", "Raiders" if distance_2d(star_wars, raiders) < distance_2d(star_wars, mean_girls) else "Mean Girls")


## 2. Distance in *n* dimensions


In [ ]:
star_wars_3 = [125, 1977, 11_000_000]
raiders_3 = [115, 1981, 18_000_000]
mean_girls_3 = [97, 2004, 17_000_000]

def distance(a, b):
    squared = 0.0
    for i in range(len(a)):
        squared += (a[i] - b[i]) ** 2
    return squared ** 0.5

print(distance(star_wars_3, raiders_3))
print(distance(star_wars_3, mean_girls_3))


### Task 2.2 — NumPy alternate


In [ ]:
def distance_np(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b))

print(distance_np(star_wars_3, raiders_3))
print("match loop?", np.isclose(distance(star_wars_3, raiders_3), distance_np(star_wars_3, raiders_3)))


## 3. Min-max normalization


In [ ]:
release_dates = [1897.0, 1998.0, 2000.0, 1948.0, 1962.0,
                 1950.0, 1975.0, 1960.0, 2017.0, 1937.0]

def min_max_normalize(lst):
    minimum = min(lst)
    maximum = max(lst)
    return [(v - minimum) / (maximum - minimum) for v in lst]

print(min_max_normalize(release_dates))
print("1897 →", min_max_normalize(release_dates)[0], "(the earliest year, so it sits at 0)")


## 4. Movie data + from-scratch classifier


In [ ]:
movies = pd.read_csv("data/knn_movies.csv")
movie_dataset = {
    row.title: [float(row.duration), float(row.year), float(row.budget)]
    for row in movies.itertuples()
}
movie_labels = {row.title: int(row.good) for row in movies.itertuples()}
print("n movies:", len(movie_dataset))
print("sample:", list(movie_dataset.items())[:2])


In [ ]:
def fit_minmax(dataset):
    X = np.array(list(dataset.values()), dtype=float)
    return X.min(axis=0), X.max(axis=0)

def apply_minmax(point, mins, maxs):
    point = np.asarray(point, dtype=float)
    return ((point - mins) / (maxs - mins)).tolist()

def normalize_dataset(dataset):
    mins, maxs = fit_minmax(dataset)
    return {t: apply_minmax(v, mins, maxs) for t, v in dataset.items()}, mins, maxs

movie_dataset_n, mins, maxs = normalize_dataset(movie_dataset)
print("mins:", mins)
print("maxs:", maxs)


In [ ]:
def classify(unknown, dataset, labels, k):
    distances = []
    for title, point in dataset.items():
        distances.append([distance(unknown, point), title])
    distances.sort()
    neighbors = distances[:k]
    num_good = sum(1 for _, title in neighbors if labels[title] == 1)
    num_bad = k - num_good
    return 1 if num_good > num_bad else 0


In [ ]:
title = "Call Me By Your Name"
print("already in set?", title in movie_dataset)
held_out = {t: p for t, p in movie_dataset_n.items() if t != title}
held_labels = {t: lab for t, lab in movie_labels.items() if t != title}
raw = movie_dataset.get(title, [132, 2017, 3_500_000])
unknown_n = apply_minmax(raw, mins, maxs)
pred = classify(unknown_n, held_out, held_labels, 5)
print("normalized:", unknown_n)
print("k=5 vote (1=good):", pred)


## 5. Alternate neighbor search


In [ ]:
def classify_np(unknown, X, y, k):
    d = np.linalg.norm(X - np.asarray(unknown, dtype=float), axis=1)
    idx = np.argsort(d)[:k]
    votes = y[idx]
    return int(votes.sum() > k / 2)

titles = list(movie_dataset_n.keys())
X_m = np.array([movie_dataset_n[t] for t in titles])
y_m = np.array([movie_labels[t] for t in titles])
print("numpy vote:", classify_np(unknown_n, X_m, y_m, 5))

# sklearn NearestNeighbors alternate
nn = NearestNeighbors(n_neighbors=5, metric="euclidean")
nn.fit(X_m)
_, ind = nn.kneighbors([unknown_n])
print("sklearn neighbors:", [titles[i] for i in ind[0]])


In [ ]:
def classify_manhattan(unknown, X, y, k):
    d = np.abs(X - np.asarray(unknown, dtype=float)).sum(axis=1)
    idx = np.argsort(d)[:k]
    return int(y[idx].sum() > k / 2)

print("Manhattan vote:", classify_manhattan(unknown_n, X_m, y_m, 5))


## 6. Breast-cancer data


In [ ]:
breast_cancer_data = load_breast_cancer()
print("first row (8 feats):", breast_cancer_data.data[0, :8])
print("feature_names[:8]:", list(breast_cancer_data.feature_names[:8]))
print("target[:10]:", breast_cancer_data.target[:10])
print("target_names:", breast_cancer_data.target_names)
print("first point class:", breast_cancer_data.target_names[breast_cancer_data.target[0]])
# csv path (same numbers)
df_c = pd.read_csv("data/knn_cancer.csv")
print("csv shape:", df_c.shape)


In [ ]:
y_all = breast_cancer_data.target
print("malignant (0):", int((y_all == 0).sum()), "benign (1):", int((y_all == 1).sum()))
print("prevalence benign:", y_all.mean())


## 7. Train / validation split


In [ ]:
training_data, validation_data, training_labels, validation_labels = train_test_split(
    breast_cancer_data.data,
    breast_cancer_data.target,
    test_size=0.2,
    random_state=100,
    stratify=breast_cancer_data.target,
)
print("train", len(training_data), "valid", len(validation_data))
print("label lengths match?", len(training_data) == len(training_labels))


## 8. KNeighborsClassifier


In [ ]:
classifier = KNeighborsClassifier(n_neighbors=3)
classifier.fit(training_data, training_labels)
acc_raw = classifier.score(validation_data, validation_labels)
print("k=3 unscaled valid acc:", acc_raw)


In [ ]:
scaler = MinMaxScaler()
Xtr_s = scaler.fit_transform(training_data)
Xva_s = scaler.transform(validation_data)
clf_s = KNeighborsClassifier(n_neighbors=3)
clf_s.fit(Xtr_s, training_labels)
acc_s = clf_s.score(Xva_s, validation_labels)
print("k=3 scaled valid acc:", acc_s)


## 9. Sweep *k* and graph


In [ ]:
k_list = list(range(1, 51))
accuracies = []
for k in k_list:
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(Xtr_s, training_labels)
    accuracies.append(clf.score(Xva_s, validation_labels))
best_i = int(np.argmax(accuracies))
print("best k, acc:", k_list[best_i], accuracies[best_i])
print("k=1:", accuracies[0], " k=3:", accuracies[2], " k=15:", accuracies[14])


In [ ]:
plt.figure(figsize=(8.4, 4.2))
plt.plot(k_list, accuracies, color="#1A5276")
plt.axvline(k_list[best_i], color="#117A65", ls="--", label=f"best k={k_list[best_i]}")
plt.xlabel("k")
plt.ylabel("Validation Accuracy")
plt.title("Breast Cancer Classifier Accuracy")
plt.legend()
plt.show()


## 10. More practice


In [ ]:
blob = pd.read_csv("data/knn_2d.csv")
Xb = blob[["x1", "x2"]].to_numpy()
yb = blob["y"].to_numpy()
Xtrb, Xvab, ytrb, yvab = train_test_split(Xb, yb, test_size=0.2, random_state=7, stratify=yb)

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8))
xx, yy = np.meshgrid(
    np.linspace(Xb[:, 0].min() - 0.4, Xb[:, 0].max() + 0.4, 200),
    np.linspace(Xb[:, 1].min() - 0.4, Xb[:, 1].max() + 0.4, 200),
)
grid = np.c_[xx.ravel(), yy.ravel()]
for ax, k in zip(axes, [1, 21]):
    clf = KNeighborsClassifier(n_neighbors=k).fit(Xtrb, ytrb)
    zz = clf.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=["#FADBD8", "#D6EAF8"])
    ax.scatter(Xtrb[ytrb == 0, 0], Xtrb[ytrb == 0, 1], c="#C0392B", s=14)
    ax.scatter(Xtrb[ytrb == 1, 0], Xtrb[ytrb == 1, 1], c="#1A5276", s=14)
    ax.set_title(f"k={k}  valid acc={clf.score(Xvab, yvab):.3f}")
plt.suptitle("2-D practice: small k hugs every point")
plt.tight_layout()
plt.show()


In [ ]:
loans = pd.read_csv("data/knn_loans.csv")
Xl = loans[["dti", "utilization"]].to_numpy()
yl = loans["default"].to_numpy()
Xtrl, Xval, ytrl, yval = train_test_split(Xl, yl, test_size=0.25, random_state=21, stratify=yl)
sc_l = MinMaxScaler()
Xtrl_s = sc_l.fit_transform(Xtrl)
Xval_s = sc_l.transform(Xval)
for k in (1, 5, 15):
    clf = KNeighborsClassifier(n_neighbors=k).fit(Xtrl_s, ytrl)
    pred = clf.predict(Xval_s)
    print(f"k={k:2d}  acc={accuracy_score(yval, pred):.3f}  cm={confusion_matrix(yval, pred).tolist()}")


## 11. Simulation


In [ ]:
# ----- editable -----
K_FIXED = 15
NOISE = 0.00
TRAIN_FRAC = 1.00
N_EXTRA = 0
RANDOM_STATE = 100
# --------------------

bc = load_breast_cancer()
X, y = bc.data, bc.target
Xtr, Xva, ytr, yva = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
sc = MinMaxScaler()
Xtr = sc.fit_transform(Xtr)
Xva = sc.transform(Xva)

rng = np.random.default_rng(RANDOM_STATE)
m = max(K_FIXED + 1, int(len(Xtr) * TRAIN_FRAC))
idx = rng.choice(len(Xtr), size=m, replace=False)
Xtr, ytr = Xtr[idx], ytr[idx].copy()
flip = rng.random(len(ytr)) < NOISE
ytr[flip] = 1 - ytr[flip]
if N_EXTRA > 0:
    Xtr = np.hstack([Xtr, rng.normal(size=(len(Xtr), N_EXTRA))])
    Xva = np.hstack([Xva, rng.normal(size=(len(Xva), N_EXTRA))])
    sc2 = MinMaxScaler()
    Xtr = sc2.fit_transform(Xtr)
    Xva = sc2.transform(Xva)

clf = KNeighborsClassifier(n_neighbors=min(K_FIXED, len(Xtr)))
clf.fit(Xtr, ytr)
acc = clf.score(Xva, yva)
print(f"valid acc = {acc:.4f}   (k={K_FIXED}, noise={NOISE}, n_train={len(Xtr)}, extra={N_EXTRA})")
print("try: K_FIXED=1, NOISE=0.2, TRAIN_FRAC=0.2, N_EXTRA=80")


## 12. Audience rewrite


In [ ]:
analyst = (
    "Scaled Euclidean KNN on the 30-feature Wisconsin matrix, 80/20 stratified split "
    "(random_state=100), reaches 0.965 at k=3 and 0.974 at k=15 on 114 hold-out cells. "
    "k=1 overfits local noise; padding 80 N(0,1) columns drops accuracy into the low 0.91s "
    "(curse of dimensionality). Min-max fitted on train only. Report sensitivity to k, "
    "label flip rate, and n_train before locking a neighbor count."
)
director = (
    "A simple nearest-neighbor screen, using the usual nuclear-grade measurements from "
    "the FNA report, correctly tagged about 97 in 100 held-out slides when we compared "
    "each new cell to its 15 most similar past cases. It is a triage aid, not a diagnosis. "
    "Very small neighbor counts chase outliers; dumping extra uninformative lab fields "
    "makes look-alikes harder to find. Pathology still signs the case."
)
patient = (
    "Doctors can compare a new biopsy to past biopsies that look similar under the microscope. "
    "In a test set of 114 samples the method agreed with the recorded result about 97% of the time. "
    "It is one extra check, not a verdict on its own. Your care team uses many tests together."
)
friend = (
    "Imagine lining up past lab reports that look most like yours and taking a majority vote "
    "on what those similar cases turned out to be. With a few dozen neighbors the vote was "
    "right about 97 times out of 100 on leftover samples. Too few neighbors copies flukes; "
    "too many just repeats the overall average."
)
print(analyst); print(); print(director); print(); print(patient); print(); print(friend)


## Done

Reusable pattern: `KNN_Cancer_Reusable_Template.ipynb`. Charts live next to this file (`knn_cancer_*.png`).
